# Introduction
## Diffusion Equation
The 1D diffusion equation is given by
$$\frac{\partial u(x,t)}{\partial t} = \nu \frac{\partial^2 u(x,t)}{\partial x^2},$$
where
- $u(x,t)$ is the function we're tracking (velocity, temperature, etc)
- $x$ is position
- $t$ is time
- $\nu$ is the diffusion coefficient, ie how fast things spread

## Discretised Laplacian
The central second difference approximation
$$
\frac{d^2u(x_i)}{dx^2}\approx\frac{u_{i+1}-2u_i+u_{i-1}}{h^2}
$$
is used to form the Laplacian matrix
$$
L = \frac{1}{h^2}\begin{pmatrix}
-2 & 1 & 0 & 0 & 0 \\
1 & -2 & 1 & 0 & 0 \\
0 & 1 & -2 & 1 & 0 \\ 
0 & 0 & 1 & -2 & 1 \\
0 & 0 & 0 & 1 & -2
\end{pmatrix}
$$
where we have assumed dirichlet (non-periodic) boundary conditions.

## Time-Stepping Operator
The time-stepping operator
$$
A = \mathbb I + \nu dt \  L
$$
is the operator that carries out the time evolution in steps using the Forward Euler scheme.

## Time Evolution using Tensor Networks
In the dense case, the state vector is updated by repeatedly applying the time-step operator
$$
\mathbf u(t_{i+1})=A  \mathbf u(t_i).
$$

In the tensor network formulation, $A$ and $\mathbf u(t_0)$ are converted into an MPO and an MPS respectively. The same update rule is performed in the tensor network language, which is to contract the MPO with the MPS to obtain the next update.

After each update, the resulting MPS is compressed via SVD truncation to limit bond dimension.

In [3]:
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
import quimb.tensor as qtn
import time

# Parameters

In [16]:
# general parameters

n = 25                 # you will have a 1D line of 2^n points, and the MPSs/MPOs will have n sites.
nu = 1e-3               # diffusion coefficient 
steps = 2000            # number of steps required for time evolution
save_every = 200        # after this many steps, take a snapshot of the function to plot on the graph
cfl = 0.1               # controls time step relative to grid spacing. affects stability of time0-step scheme

u0_type = "quadratic"   # "sines" | "quadratic" | "random"

In [17]:
# TN-specific parameters

# parameters for forming the initial MPS
init_cutoff = 1e-12
init_max_bond = 64

# parameters for forming the initial MPO
mpo_cutoff = 1e-12
mpo_max_bond = 256

# SVD truncation and bond dimension limits during contractions
tn_cutoff = 1e-10
tn_max_bond = 64

# Helper Functions

In [ ]:
# ============
# DENSE SOLVER
# ============

def laplacian_1d_sparse(N, dx):
    main = -2 * np.ones(N)
    off  = 1 * np.ones(N - 1)
    L = sp.diags(
        [off, main, off],
        offsets=[-1, 0, 1],
        format="csr"
    )
    return L / (dx * dx)

N = 2**n
x = np.linspace(0, 1, N, endpoint=False)
dx = x[1] - x[0] # corresponds to h in mathematical notation
dt = cfl * dx*dx / nu 
alpha = nu * dt / (dx * dx)

if u0_type == "sines":
    u0 = np.sin(2*np.pi*2*x) + 0.5*np.sin(2*np.pi*7*x)
elif u0_type == "quadratic":
    u0 = x**2
elif u0_type == "random":
    u0 = np.random.randn(N)
else:
    u0 = np.asarray(u0_type)


def evolve_dense(u0, steps, A, dt, save_every=50):
    u = u0.copy()

    saved = []
    times = []
    norms = []

    t = 0.0
    
    for i in range(steps):
        # save current state
        if i % save_every == 0:
            saved.append(u.copy())
            times.append(t)
            norms.append(np.linalg.norm(u))
    
        # advance one timestep
        u = A @ u
        t += dt
    
    # save final state
    saved.append(u.copy())
    times.append(t)
    norms.append(np.linalg.norm(u))
    
    return np.array(times), np.array(saved), np.array(norms)



# ==================
# MPS/MPO GENERATION
# ==================

# these helper functions convert from vectors to MPS and vice versa,
# as well as from matrices to MPOs

def vec_to_qtt_mps(u, n, cutoff=1e-10, max_bond=64):
    T = np.asarray(u).reshape((2,) * n)
    T = T.transpose(tuple(range(n - 1, -1, -1)))
    # reverses MPS direction such that least significant bit is first: easier for shift MPOs later
    return qtn.MatrixProductState.from_dense(T, cutoff=cutoff, max_bond=max_bond)


def qtt_identity_mpo(n):
    W = np.zeros((1, 1, 2, 2))
    W[0, 0] = np.eye(2)
    arrays = [W.copy() for _ in range(n)]
    return qtn.MatrixProductOperator(arrays, shape='lrud')


def qtt_shift_plus_mpo(n):
    arrays = []
    # first site: incoming carry fixed to 1
    W0 = np.zeros((1, 2, 2, 2))
    for carry_out in (0, 1):
        for x in (0, 1):
            s = x + 1
            y = s & 1
            c = (s >> 1) & 1
            if c == carry_out:
                W0[0, carry_out, y, x] = 1.0
    arrays.append(W0)

    # middle sites: propagate carry
    W = np.zeros((2, 2, 2, 2))
    for carry_in in (0, 1):
        for x in (0, 1):
            s = x + carry_in
            y = s & 1
            carry_out = (s >> 1) & 1
            W[carry_in, carry_out, y, x] = 1.0
    for _ in range(n - 2):
        arrays.append(W.copy())

    # last site: require no overflow (carry_out = 0)
    WN = np.zeros((2, 1, 2, 2))
    for carry_in in (0, 1):
        for x in (0, 1):
            s = x + carry_in
            y = s & 1
            carry_out = (s >> 1) & 1
            if carry_out == 0:
                WN[carry_in, 0, y, x] = 1.0
    arrays.append(WN)

    return qtn.MatrixProductOperator(arrays, shape='lrud')


def qtt_shift_minus_mpo(n):
    arrays = []
    # first site: incoming borrow fixed to 1
    W0 = np.zeros((1, 2, 2, 2))
    for borrow_out in (0, 1):
        for x in (0, 1):
            d = x - 1
            if d >= 0:
                y = d
                b = 0
            else:
                y = d + 2
                b = 1
            if b == borrow_out:
                W0[0, borrow_out, y, x] = 1.0
    arrays.append(W0)
    # middle sites: propagate borrow
    W = np.zeros((2, 2, 2, 2))
    for borrow_in in (0, 1):
        for x in (0, 1):
            d = x - borrow_in
            if d >= 0:
                y = d
                borrow_out = 0
            else:
                y = d + 2
                borrow_out = 1
            W[borrow_in, borrow_out, y, x] = 1.0
    for _ in range(n - 2):
        arrays.append(W.copy())
    # last site: require no underflow (borrow_out = 0)
    WN = np.zeros((2, 1, 2, 2))
    for borrow_in in (0, 1):
        for x in (0, 1):
            d = x - borrow_in
            if d >= 0:
                y = d
                borrow_out = 0
            else:
                y = d + 2
                borrow_out = 1
            if borrow_out == 0:
                WN[borrow_in, 0, y, x] = 1.0
    arrays.append(WN)
    return qtn.MatrixProductOperator(arrays, shape='lrud')

def qtt_diffusion_mpo(n, alpha, cutoff=1e-12, max_bond=64):
    I  = qtt_identity_mpo(n)
    Sp = qtt_shift_plus_mpo(n)
    Sm = qtt_shift_minus_mpo(n)

    A = (1.0 - 2.0 * alpha) * I + alpha * Sp + alpha * Sm
    A.compress(cutoff=cutoff, max_bond=max_bond)
    return A


# ====================
# TIME EVOLUTION IN TN
# ====================

def step_mps(mps, mpo, cutoff=1e-10, max_bond=64):
    mps_new = mpo.apply(mps)
    mps_new.compress(cutoff=cutoff, max_bond=max_bond)
    return mps_new

def evolve_mps(mps0, mpoA, steps, save_every=50, cutoff=1e-10, max_bond=64):
    mps = mps0.copy()
    saved = []
    bonds = []
    
    for i in range(steps):
        if i % save_every == 0:
            saved.append(mps.copy())
            bonds.append(max(mps.bond_sizes()))
    
        mps = step_mps(mps, mpoA, cutoff, max_bond)
    
    # save final state
    saved.append(mps.copy())
    bonds.append(max(mps.bond_sizes()))
    return saved, bonds



# =======================================
# WRAPPER FUNCTIONS TO MEASURE TIME TAKEN
# =======================================

def time_mps_construction(u0, n, cutoff, max_bond):
    t0 = time.perf_counter()
    mps0 = vec_to_qtt_mps(u0, n, cutoff, max_bond)
    t1 = time.perf_counter()
    return mps0, t1 - t0

def time_mpo_construction(n, alpha, mpo_cutoff, mpo_max_bond):
    t0 = time.perf_counter()
    mpoA = qtt_diffusion_mpo(n, alpha, cutoff=mpo_cutoff, max_bond=mpo_max_bond)
    t1 = time.perf_counter()
    return mpoA, t1 - t0

def time_evolve_dense(u0, steps, A, dt, save_every=50):
    t0 = time.perf_counter()
    evolve_dense(u0, steps, A, dt, save_every)
    t1 = time.perf_counter()
    return t1 - t0

def time_evolve_mps(mps0, mpoA, steps, save_every=50, cutoff=1e-10, max_bond=64):
    t0 = time.perf_counter()
    evolve_mps(mps0, mpoA, steps, save_every, cutoff, max_bond)
    t1 = time.perf_counter()
    return t1 - t0

# Set Up

In [19]:
N = 2**n
x = np.linspace(0, 1, N, endpoint=False)
dx = x[1] - x[0] # corresponds to h in mathematical notation
dt = cfl * dx*dx / nu 
alpha = nu * dt / (dx * dx)

if u0_type == "sines":
    u0 = np.sin(2*np.pi*2*x) + 0.5*np.sin(2*np.pi*7*x)
elif u0_type == "quadratic":
    u0 = x**2
elif u0_type == "random":
    u0 = np.random.randn(N)
else:
    u0 = np.asarray(u0_type)

L = laplacian_1d_sparse(N, dx)                             
A = sp.eye(N, format="csr") + dt * nu * L   

# Experiment
## i. Time Taken to construct MPS and MPO

In [20]:
mps0, t_mps = time_mps_construction(u0, n, init_cutoff, init_max_bond)
mpoA, t_mpo = time_mpo_construction(n, alpha, mpo_cutoff, mpo_max_bond)

print(f"MPS construction: {t_mps:.6f} s")
print(f"MPO construction: {t_mpo:.6f} s")

MPS construction: 2.136551 s
MPO construction: 0.033861 s


## ii. Time taken to run one dt

In [21]:
t_evolve_dense = time_evolve_dense(u0, 1, A, dt, save_every)
t_evolve_mps = time_evolve_mps(mps0, mpoA, 1, save_every, tn_cutoff, tn_max_bond)

print(f"Dense evolution: {t_evolve_dense:.6f} s")
print(f"TN evolution   : {t_evolve_mps:.6f} s")

Dense evolution: 0.739992 s
TN evolution   : 0.045542 s
